# OCR Evaluation on Weave — Model Comparison

Runs a [Weave Evaluation](https://weave-docs.wandb.ai/guides/core-types/evaluations) over the published `ocr-eval` dataset, comparing vision models with the **prompt held constant**:

- **Prompt**: the exact OCR prompt used by Kateb ([jbutt-wandb/ocr_handwritten_notes](https://github.com/jbutt-wandb/ocr_handwritten_notes)) (`build_ocr_prompt(contains_latex=True, contains_diagrams=True)`), sourced from the Weave prompt assets published in this project, with constants vendored from `backend/prompts/ocr_prompts.py` as fallback
- **Models**: the app's registry of vision models on [W&B Inference](https://docs.wandb.ai/guides/inference/) — Kimi K2.6 (app default), Gemma 4 31B, Qwen 3.6 35B A3B
- **Predict path**: faithful to the app's `OCRModel.predict` — 1600 px downscale, strict JSON-schema response format, `clean_markdown` post-processing
- **Scoring**: [jiwer](https://jitsi.github.io/jiwer/) OCR metrics — WER, MER, WIL, WIP, CER — on both **raw** text and a **normalized** view (lowercase, punctuation stripped, whitespace collapsed)

In [1]:
import base64
import io
import json
import logging
import os
import re

import jiwer
import pandas as pd
import weave
from dotenv import load_dotenv
from openai import AsyncOpenAI
from PIL import Image

logging.getLogger("weave").setLevel(logging.ERROR)

load_dotenv(override=True)
ENTITY = os.environ["ENTITY"]
PROJECT = os.environ["PROJECT"]

client = weave.init(f"{ENTITY}/{PROJECT}")
print(f"Weave project: {ENTITY}/{PROJECT}")

weave: weave version 0.53.11 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: jbutt.
weave: View Weave data at https://wandb.ai/wandb-smle/kateb/weave


Weave project: wandb-smle/kateb


## Load the eval dataset

In [2]:
dataset = weave.ref(f"weave:///{ENTITY}/{PROJECT}/object/ocr-eval:latest").get()
print(f"Loaded dataset '{dataset.name}' with {len(dataset.rows)} rows")

Loaded dataset 'ocr-eval' with 7 rows


In [3]:
dataset_df = dataset.to_pandas()
dataset_df.head()

,id,image,ground_truth_text
0,IMG_2555,<PIL.PngImagePlugin.PngImageFile image mode=RG...,# Rejection Sampling\n\nInvolves curating new ...
1,IMG_2556,<PIL.PngImagePlugin.PngImageFile image mode=RG...,# Generating Completions\n\nTo generate a set ...
2,IMG_2557,<PIL.PngImagePlugin.PngImageFile image mode=RG...,There are many ways to use $R$ to select top c...
3,IMG_2558,<PIL.PngImagePlugin.PngImageFile image mode=RG...,"With the selected completions, we can perform ..."
4,page_1,<PIL.JpegImagePlugin.JpegImageFile image mode=...,## Morphing Probability distributions\n\n- Let...


## OCR model

Faithful to the app's `OCRModel.predict` (`backend/services/models.py`): prompt sent as a user text part alongside the image, images downscaled to max **1600 px** (W&B Inference rejects very large payloads), strict JSON-schema response format (`{"markdown": ...}`) with raw-content fallback, `max_tokens=32768` (Gemma/Qwen spend heavily on reasoning before answering), and the app's `clean_markdown` post-processing. `weave.Model` versions `model_name` and `prompt`, so the compare view shows the model as the only attribute that differs between runs.

In [4]:
# App constants (backend/services/inference_service.py)
MAX_IMAGE_DIMENSION = 1600
MAX_OUTPUT_TOKENS = 32_768

OCR_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "OCROutput",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {"markdown": {"type": "string"}},
            "required": ["markdown"],
            "additionalProperties": False,
        },
    },
}

In [5]:
inference_client = AsyncOpenAI(
    base_url="https://api.inference.wandb.ai/v1",
    api_key=os.environ["WANDB_API_KEY"],
    default_headers={"OpenAI-Project": f"{ENTITY}/{PROJECT}"},
)


def image_to_data_uri(image: Image.Image, max_side: int = MAX_IMAGE_DIMENSION, quality: int = 85) -> str:
    img = image.copy()
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=quality)
    return f"data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}"


def clean_markdown(text: str) -> str:
    """Remove any markdown code fences that wrap the entire content (app's post-processing)."""
    text = re.sub(r'^```(?:markdown)?\s*\n?', '', text.strip())
    text = re.sub(r'\n?```\s*$', '', text.strip())
    return text.strip()


In [6]:
class OCRModel(weave.Model):
    model_name: str
    prompt: str

    @weave.op
    async def predict(self, image: Image.Image) -> str:
        response = await inference_client.chat.completions.create(
            model=self.model_name,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": self.prompt},
                        {"type": "image_url", "image_url": {"url": image_to_data_uri(image)}},
                    ],
                }
            ],
            response_format=OCR_RESPONSE_FORMAT,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
        content = (response.choices[0].message.content or "").strip()

        try:
            markdown = json.loads(content)["markdown"]
        except Exception:
            markdown = content
        return clean_markdown(markdown)

## The constant prompt

Each prompt piece is sourced from the Weave prompt assets published in this project (`ocr-system-prompt`, `ocr-latex-instructions`, `ocr-diagram-instructions`, `ocr-output-instructions`). If an asset can't be fetched, the constants vendored from Kateb's [`backend/prompts/ocr_prompts.py`](https://github.com/jbutt-wandb/ocr_handwritten_notes/blob/main/backend/prompts/ocr_prompts.py) are used as fallback. The eval uses `build_ocr_prompt(contains_latex=True, contains_diagrams=True)` — the fullest prompt the app can build — held constant across all models.

In [ ]:
# Fallback constants — vendored verbatim from Kateb's backend/prompts/ocr_prompts.py

SYSTEM_PROMPT = """You are an expert OCR assistant specializing in converting handwritten notes to clean, well-formatted Markdown. Your task is to:

1. Accurately transcribe all handwritten text
2. Preserve the logical structure (headings, lists, paragraphs)
3. Maintain the original organization and flow of ideas
4. Use appropriate Markdown formatting (headers, bold, italics, lists)

Output ONLY the Markdown content, no explanations or commentary."""

LATEX_INSTRUCTIONS = """MATHEMATICAL EQUATIONS:
- Convert ALL mathematical expressions to LaTeX format
- Use inline math with single dollar signs: $x^2 + y^2 = r^2$
- Use display math with double dollar signs for standalone equations:
  $$\\int_{a}^{b} f(x) dx = F(b) - F(a)$$
- Properly escape special characters
- Use standard LaTeX notation:
  - Fractions: \\frac{numerator}{denominator}
  - Square roots: \\sqrt{x} or \\sqrt[n]{x}
  - Subscripts/superscripts: x_i, x^2
  - Greek letters: \\alpha, \\beta, \\gamma
  - Summations: \\sum_{i=1}^{n}
  - Integrals: \\int, \\iint, \\oint
  - Limits: \\lim_{x \\to \\infty}
- Preserve equation numbering if visible"""

DIAGRAM_INSTRUCTIONS = """GRAPHS AND DIAGRAMS:
- Describe diagrams in detail within markdown blockquotes or as descriptive text
- For graphs, describe:
  - Type of graph (line, bar, scatter, etc.)
  - Axes labels and scales
  - Key data points or trends
  - Any annotations or labels
- For diagrams, describe:
  - The overall structure and layout
  - Components and their relationships
  - Labels and text within the diagram
  - Arrows, connections, or flow directions
- Format descriptions clearly:

  > **Diagram: [Title/Description]**
  > - Component A connects to Component B
  > - Arrow indicates flow direction from X to Y

- If the diagram contains mathematical relationships, express them in LaTeX"""

OUTPUT_INSTRUCTIONS = """OUTPUT FORMAT:
- Return clean, properly formatted Markdown
- Use appropriate heading levels (# ## ###)
- Preserve bullet points and numbered lists
- Separate distinct sections with blank lines
- Do not include any preamble or explanation
- Start directly with the transcribed content
- Do NOT wrap output in code fences (no ```markdown```)
- Do NOT add page dividers or separators
- Output raw markdown text only"""


def build_ocr_prompt(
    contains_latex: bool,
    contains_diagrams: bool,
    custom_instructions: str = "",
) -> str:
    """Build a dynamic OCR prompt based on user options."""
    prompt_parts = [SYSTEM_PROMPT]
    if contains_latex:
        prompt_parts.append(LATEX_INSTRUCTIONS)
    if contains_diagrams:
        prompt_parts.append(DIAGRAM_INSTRUCTIONS)
    prompt_parts.append(OUTPUT_INSTRUCTIONS)
    if custom_instructions and custom_instructions.strip():
        prompt_parts.append(f"ADDITIONAL USER INSTRUCTIONS:\n{custom_instructions.strip()}")
    return "\n\n".join(prompt_parts)


# --- End vendored code ---


def fetch_prompt(asset_name: str, fallback: str) -> str:
    """Fetch a prompt piece from its published Weave asset, falling back to the vendored constant."""
    try:
        content = weave.ref(f"weave:///{ENTITY}/{PROJECT}/object/{asset_name}:latest").get().content
        print(f"Sourced '{asset_name}' from Weave")
        return content
    except Exception as e:
        print(f"Could not fetch '{asset_name}' from Weave ({e}); using vendored fallback")
        return fallback


SYSTEM_PROMPT = fetch_prompt("ocr-system-prompt", SYSTEM_PROMPT)
LATEX_INSTRUCTIONS = fetch_prompt("ocr-latex-instructions", LATEX_INSTRUCTIONS)
DIAGRAM_INSTRUCTIONS = fetch_prompt("ocr-diagram-instructions", DIAGRAM_INSTRUCTIONS)
OUTPUT_INSTRUCTIONS = fetch_prompt("ocr-output-instructions", OUTPUT_INSTRUCTIONS)

OCR_PROMPT = build_ocr_prompt(contains_latex=True, contains_diagrams=True)

# The app's model registry (backend/services/inference_service.py::AVAILABLE_MODELS)
MODELS = {
    "moonshotai/Kimi-K2.6": "kimi-k2.6",
    "google/gemma-4-31B-it": "gemma-4-31b",
    "Qwen/Qwen3.6-35B-A3B": "qwen3.6-35b",
    "deepseek-ai/DeepSeek-V4.1-Flash": "DeepSeek-V4.1-Flash",
}

print(f"Prompt: {len(OCR_PROMPT)} chars | Models: {list(MODELS.values())}")

Sourced 'ocr-system-prompt' from Weave
Sourced 'ocr-latex-instructions' from Weave
Sourced 'ocr-diagram-instructions' from Weave
Sourced 'ocr-output-instructions' from Weave
Prompt: 2152 chars | Models: ['kimi-k2.6', 'gemma-4-31b', 'qwen3.6-35b', 'deepseek-ai/DeepSeek-V4.1-Flash']


## jiwer scorer

Returns every metric twice: `raw` (exact text, LaTeX/markdown included) and `normalized` (lowercase, punctuation stripped, whitespace collapsed — content accuracy without formatting noise). Weave averages the numeric leaves across rows in the eval summary.

In [8]:
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def jiwer_metrics(reference: str, hypothesis: str) -> dict:

    if not hypothesis.strip():
        return {"wer": 1.0, "mer": 1.0, "wil": 1.0, "wip": 0.0, "cer": 1.0}

    words = jiwer.process_words(reference, hypothesis)
    chars = jiwer.process_characters(reference, hypothesis)

    return {
        "wer": words.wer,
        "mer": words.mer,
        "wil": words.wil,
        "wip": words.wip,
        "cer": chars.cer,
    }


@weave.op
def ocr_metrics(ground_truth_text: str, output: str) -> dict:
    return {
        "raw": jiwer_metrics(ground_truth_text, output),
        "normalized": jiwer_metrics(normalize_text(ground_truth_text), normalize_text(output)),
    }

## Run the model comparison

One `Evaluation` (same dataset, same scorer), evaluated once per model with the prompt held constant — M models × N images = MN inference calls.

In [9]:
evaluation = weave.Evaluation(
    name="ocr-model-comparison", 
    dataset=dataset, 
    scorers=[ocr_metrics]
    )

In [10]:
MODELS

{'moonshotai/Kimi-K2.6': 'kimi-k2.6',
 'google/gemma-4-31B-it': 'gemma-4-31b',
 'Qwen/Qwen3.6-35B-A3B': 'qwen3.6-35b',
 'DeepSeek V4.1-Flash': 'deepseek-ai/DeepSeek-V4.1-Flash'}

In [11]:
# Run evaluation for each model and store results
results = {}

for model_id, slug in MODELS.items():

    model = OCRModel(
        name=f"ocr-{slug}",
        model_name=model_id,
        prompt=OCR_PROMPT,
    )

    print(f"Evaluating model: {model_id} ...")

    results[model_id] = await evaluation.evaluate(
        model,
        __weave={"display_name": f"ocr-{slug}"}
    )

print("\nAll eval runs complete.")
print(f"Compare them at: https://wandb.ai/{ENTITY}/{PROJECT}/weave/evaluations")

Evaluating model: moonshotai/Kimi-K2.6 ...


weave: 🍩 https://wandb.ai/wandb-smle/kateb/r/call/01a11623-6937-7613-bdb3-b1249cc3bec1
weave: Evaluated 1 of 7 examples
weave: Evaluated 2 of 7 examples
weave: Evaluated 3 of 7 examples
weave: Evaluated 4 of 7 examples
weave: Evaluated 5 of 7 examples
weave: Evaluated 6 of 7 examples
weave: Evaluated 7 of 7 examples
weave: Evaluation summary {
weave:   "ocr_metrics": {
weave:     "raw": {
weave:       "wer": {
weave:         "mean": 0.2525779535000213
weave:       },
weave:       "mer": {
weave:         "mean": 0.2508394149718546
weave:       },
weave:       "wil": {
weave:         "mean": 0.32478766972571904
weave:       },
weave:       "wip": {
weave:         "mean": 0.675212330274281
weave:       },
weave:       "cer": {
weave:         "mean": 0.23364726346648657
weave:       }
weave:     },
weave:     "normalized": {
weave:       "wer": {
weave:         "mean": 0.26129611869743324
weave:       },
weave:       "mer": {
weave:         "mean": 0.2608638021725664
weave:       },
weave:

Evaluating model: google/gemma-4-31B-it ...


weave: 🍩 https://wandb.ai/wandb-smle/kateb/r/call/01a11627-3fea-7c5b-bcd3-be746ab5caa9
weave: Evaluated 1 of 7 examples
weave: Evaluated 2 of 7 examples
weave: Evaluated 3 of 7 examples
weave: Evaluated 4 of 7 examples
weave: Evaluated 5 of 7 examples
weave: Evaluated 6 of 7 examples
weave: Evaluated 7 of 7 examples
weave: Evaluation summary {
weave:   "ocr_metrics": {
weave:     "raw": {
weave:       "wer": {
weave:         "mean": 17.98734183794198
weave:       },
weave:       "mer": {
weave:         "mean": 0.5017787083188271
weave:       },
weave:       "wil": {
weave:         "mean": 0.5630697468140136
weave:       },
weave:       "wip": {
weave:         "mean": 0.4369302531859864
weave:       },
weave:       "cer": {
weave:         "mean": 19.5568363118409
weave:       }
weave:     },
weave:     "normalized": {
weave:       "wer": {
weave:         "mean": 10.956667052316813
weave:       },
weave:       "mer": {
weave:         "mean": 0.4614443514913112
weave:       },
weave:     

Evaluating model: Qwen/Qwen3.6-35B-A3B ...


weave: 🍩 https://wandb.ai/wandb-smle/kateb/r/call/01a11637-a79a-7c78-9efc-ccac45026017
weave: Evaluated 1 of 7 examples
weave: Evaluated 2 of 7 examples
weave: Evaluated 3 of 7 examples
weave: Evaluated 4 of 7 examples
weave: Evaluated 5 of 7 examples
weave: Evaluated 6 of 7 examples
weave: Evaluated 7 of 7 examples
weave: Evaluation summary {
weave:   "ocr_metrics": {
weave:     "raw": {
weave:       "wer": {
weave:         "mean": 0.2072128798317448
weave:       },
weave:       "mer": {
weave:         "mean": 0.20136515110051928
weave:       },
weave:       "wil": {
weave:         "mean": 0.3250914102002916
weave:       },
weave:       "wip": {
weave:         "mean": 0.6749085897997084
weave:       },
weave:       "cer": {
weave:         "mean": 0.15199229881554524
weave:       }
weave:     },
weave:     "normalized": {
weave:       "wer": {
weave:         "mean": 0.18049169451999345
weave:       },
weave:       "mer": {
weave:         "mean": 0.17238191571552558
weave:       },
weav

Evaluating model: DeepSeek V4.1-Flash ...


weave: model_output failed
weave: Traceback (most recent call last):
weave:   File "/Users/jbutt/Developer/make_ocr_dataset/.venv/lib/python3.14/site-packages/weave/flow/model.py", line 143, in apply_model_async
weave:     model_output, model_call = await async_call_op(
weave:                                ^^^^^^^^^^^^^^^^^^^^
weave:         model_predict_op, **model_predict_args
weave:         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
weave:     )
weave:     ^
weave:   File "/Users/jbutt/Developer/make_ocr_dataset/.venv/lib/python3.14/site-packages/weave/trace/op.py", line 692, in _call_async_func
weave:     res = await func(*args, **kwargs)
weave:           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
weave:   File "/var/folders/dm/kyhk1vhx7jb_k1b2z7zdns6c0000gp/T/ipykernel_61400/3200228565.py", line 7, in predict
weave:     response = await inference_client.chat.completions.create(
weave:                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
weave:     ...<12 lines>...
weave:     )
weave:    


All eval runs complete.
Compare them at: https://wandb.ai/wandb-smle/kateb/weave/evaluations


weave: 🍩 https://wandb.ai/wandb-smle/kateb/r/call/01a11639-1f6f-7407-b955-c8bf72efc07a


## Model leaderboard

Mean metrics per model, sorted by normalized WER (lower is better for WER/MER/WIL/CER; higher is better for WIP).

In [12]:
import weave
from weave.flow import leaderboard
from weave.trace.ref_util import get_ref

# Pull evaluation results into a Weave table for further analysis
evaluations = client.get_evaluations()

# Check evaluation names
for eval in evaluations:
    print(eval.name)

filtered_eval = [i for i in evaluations if i.name == 'ocr-model-comparison'][0]

# Define the leaderboard
spec = leaderboard.Leaderboard(
    name="OCR Model Comparison Leaderboard",
    description="Comparing models on the OCR evaluation dataset using WER, MER, WIL, WIP, and CER metrics.",
    columns=[
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.cer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.wer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.mer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.wil.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="raw.wip.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.cer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.wer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.mer.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.wil.mean",          # e.g. "mean", "true_fraction"
        ),
        leaderboard.LeaderboardColumn(
            evaluation_object_ref=get_ref(filtered_eval).uri(),
            scorer_name="ocr_metrics",       # name of the scoring function used
            summary_metric_path="normalized.wip.mean",          # e.g. "mean", "true_fraction"
        ),
    ]
)

# Publish the leaderboard
weave.publish(spec)

# Optionally retrieve and print results
results = leaderboard.get_leaderboard_results(spec, client)

weave: 📦 Published to https://wandb.ai/wandb-smle/kateb/weave/leaderboards/OCR-Model-Comparison-Leaderboard


ocr-model-comparison
